# Hand-correct zero-crossing segmentation

Queue: every dog-run flagged by the β<sub>ω</sub> screen (`hand_check` in
`results/analysis/force_phase_regression.csv`, β<sub>ω</sub> outside
[−0.77, −0.33]) that has no saved correction yet. Saved files go to
`data/curated/hand_corrected/` and are picked up automatically by
`scripts/05_run_analysis.py`.

The signal shown is exactly what the pipeline segments (0.5 Hz high-pass,
z-scored `az`), cropped to the dog's qualifying windows plus 20 s padding.
Grey = outside any qualifying window, blue = below 3 m/s; neither reaches the
analysis, so they don't need precise correction.

**Per dog:** run the cells in order. Correct in the popup (left-click = add,
'd' = delete mode, 'a' = add mode, 's' = save, 'z' = zoom to full), close the
window, run the verify cell, then re-run from the config cell for the next
dog. To do a specific dog, overwrite `DAY`/`DOG` after the config cell.

In [ ]:
%matplotlib qt

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "tools" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "tools"))

import numpy as np
import pandas as pd

from segmentation_gui import load_for_editing, ZeroCrossingEditor, verify_saved
from synchrony import config, datasets as ds
from synchrony.analysis import hand_corrected_path
from synchrony.windows import qualifying_ranges


def flagged_queue():
    f = pd.read_csv(config.ANALYSIS_DIR / "force_phase_regression.csv")
    f = f[f.hand_check == True]  # noqa: E712 -- NaN where no regression was fitted
    return [(r.day, r.dog, r.beta_omega) for r in f.itertuples()]


for day, dog, b in flagged_queue():
    print(f"{'[x]' if hand_corrected_path(day, dog).exists() else '[ ]'} {day:18s} {dog:9s} beta_omega {b:+.2f}")

## Config: picks the next flagged dog without a saved correction

In [ ]:
todo = [(d, g) for d, g, _ in flagged_queue() if not hand_corrected_path(d, g).exists()]
if not todo:
    raise SystemExit("Nothing left: every flagged dog-run has a saved correction.")
DAY, DOG = todo[0]
print(f"Next up: {DAY} / {DOG}")

In [ ]:
DOG_FILE = dict(ds.ANALYSED_DOGS[DAY])[DOG]
GPS_FILE = ds.RUN[DAY].gps_file
SAVE_PATH = hand_corrected_path(DAY, DOG)
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

crop_to = qualifying_ranges(DAY, DOG)
highlight = qualifying_ranges(DAY, DOG, pad_s=0)
t, z, auto_zc, run_start, speed_mps, qualifying_spans = load_for_editing(
    config.RAW_DIR / DAY / DOG_FILE, config.RAW_DIR / DAY / GPS_FILE, crop_to=crop_to, highlight=highlight)
print(f"span={t[-1]:.0f}s  n_auto_crossings={len(auto_zc)}  -> {SAVE_PATH}")

# resume from an earlier partial save instead of the automatic pass
start_from = np.load(SAVE_PATH) if SAVE_PATH.exists() else auto_zc

In [ ]:
editor = ZeroCrossingEditor(t, z, zero_crossings=start_from, save_path=SAVE_PATH, speed_mps=speed_mps,
                            speed_min=ds.SPEED_MIN, qualifying_spans=qualifying_spans)

## After pressing 's' and closing the figure, confirm the save

In [ ]:
verify_saved(SAVE_PATH, auto_n=len(auto_zc))